Raw LDJSON → a clean DataFrame: dedup, sentinel handling, dtype coercion, and the `text_blob`/`cat_blob` columns the feature pipeline consumes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

Matplotlib is building the font cache; this may take a moment.


In [2]:
DATA_PATH = "../data/marketing_sample_for_amazon_com-amazon_fashion_products__20200201_20200430__30k_data.ldjson"

In [3]:
df_raw = pd.read_json(DATA_PATH, lines=True)
print(f"Raw rows: {len(df_raw):,}  |  Columns: {len(df_raw.columns)}")

Raw rows: 30,000  |  Columns: 33


In [4]:
#remove the duplicates
before = len(df_raw)
df = df_raw.drop_duplicates(subset=["uniq_id"]).copy()
after  = len(df)

In [5]:
print(f"Before: {before:,} rows")
print(f"After:  {after:,} rows")
print(f"Removed {before - after:,} duplicates")

Before: 30,000 rows
After:  30,000 rows
Removed 0 duplicates


In [6]:
#placeholder for missing weights
_WEIGHT_SENTINEL = 999_999_999

df["weight"] = pd.to_numeric(df["weight"], errors="coerce")

before_nulls = df["weight"].isna().sum()
df.loc[df["weight"] >= _WEIGHT_SENTINEL, "weight"] = np.nan
after_nulls  = df["weight"].isna().sum()

print(f"Null weight before sentinel fix: {before_nulls:,}  ({before_nulls/len(df):.1%})")
print(f"Null weight after  sentinel fix: {after_nulls:,}  ({after_nulls/len(df):.1%})")
print(f"\nEntire column is null: {df['weight'].isna().all()}")
print("→ weight will be excluded from the feature pipeline (all-NaN guard in FeatureEngine)")


Null weight before sentinel fix: 6,255  (20.8%)
Null weight after  sentinel fix: 30,000  (100.0%)

Entire column is null: True
→ weight will be excluded from the feature pipeline (all-NaN guard in FeatureEngine)


In [7]:
numeric_candidates = ["sales_price", "rating", "no__of_reviews",
                      "no__of_sellers", "discount_percentage", "parent_rank"]

numeric_cols = [c for c in numeric_candidates if c in df.columns]
print(f"Coercing columns: {numeric_cols}\n")

print(f"{'Column':<25} {'Before dtype':<15} {'After dtype':<15} {'Null % after'}")
print("-" * 65)

for col in numeric_cols:
    before_dtype = df[col].dtype
    df[col] = pd.to_numeric(df[col], errors="coerce")
    null_pct = df[col].isna().mean()
    print(f"{col:<25} {str(before_dtype):<15} {str(df[col].dtype):<15} {null_pct:.1%}")


Coercing columns: ['sales_price', 'rating', 'no__of_reviews', 'no__of_sellers', 'discount_percentage']

Column                    Before dtype    After dtype     Null % after
-----------------------------------------------------------------
sales_price               float64         float64         9.6%
rating                    float64         float64         0.0%
no__of_reviews            float64         float64         88.5%
no__of_sellers            float64         float64         96.6%
discount_percentage       object          float64         100.0%


In [8]:
for col in ["amazon_prime__y_or_n", "best_seller_tag__y_or_n"]:
    if col not in df.columns:
        continue
    df[col] = (df[col].fillna("N").str.upper() == "Y").astype(float)
    print(f"{col}: {df[col].value_counts().to_dict()}")

amazon_prime__y_or_n: {0.0: 16500, 1.0: 13500}
best_seller_tag__y_or_n: {0.0: 29987, 1.0: 13}


In [9]:
#Brand and colour might be in mixed case with whitespace
for col in ["brand", "colour"]:
    sample_before = df[col].dropna().head(3).tolist()
    df[col] = df[col].fillna("").str.lower().str.strip()
    sample_after  = df[col].head(3).tolist()
    print(f"{col}")
    print(f"  Before: {sample_before}")
    print(f"  After:  {sample_after}\n")

print(f"Unique brands after normalise:  {df['brand'].nunique():,}")
print(f"Unique colours after normalise: {df['colour'].nunique():,}")


brand
  Before: ["LA' Facon", 'LOVISTA', 'Forest Club']
  After:  ["la' facon", '', 'lovista']

colour
  Before: ['Black|Blue|Grey', 'Black|Navy Blue', 'Black|Maroon']
  After:  ['', '', '']

Unique brands after normalise:  6,339
Unique colours after normalise: 4,672


In [10]:
def extract_category(val):
    if isinstance(val, dict):
        keys = [k for k in val if k not in ("", None)]
        return str(keys[0]) if keys else ""
    return str(val) if pd.notna(val) else ""

def extract_child_category(val):
    if isinstance(val, dict):
        keys = [k for k in val if k not in ("", None)]
        return str(keys[1]) if len(keys) >= 2 else ""
    return ""

df["category"]       = df["parent___child_category__all"].apply(extract_category)
df["child_category"] = df["parent___child_category__all"].apply(extract_child_category)

print("Top 10 parent categories:")
print(df["category"].value_counts().head(10))
print(f"\nUnique parent categories: {df['category'].nunique():,}")

print("\nTop 10 child categories (subcategories — the real discriminating signal):")
print(df["child_category"].value_counts().head(10))
print(f"\nUnique child categories: {df['child_category'].nunique():,}")
print(f"Empty child_category:    {(df['child_category'] == '').sum():,} ({(df['child_category'] == '').mean():.1%})")


Top 10 parent categories:
category
ClothingAccessories      24893
                          4503
SportsFitnessOutdoors      374
BabyProducts                89
HomeKitchen                 23
AmazonLaunchpad             20
BagsWalletsandLuggage       18
HealthPersonalCare          18
CarMotorbike                16
ToysGames                   14
Name: count, dtype: int64

Unique parent categories: 20

Top 10 child categories (subcategories — the real discriminating signal):
child_category
                                5149
WomensKurtasKurtis              3534
MensT_Shirts                    3385
WomensSarees                    1808
MensFormalShirts                1060
GirlsDressesJumpsuits            992
MensCasualShirts                 866
BoysT_Shirts                     856
WomensDresses                    759
WomensEthnicUnstitchedFabric     664
Name: count, dtype: int64

Unique child categories: 219
Empty child_category:    5,149 (17.2%)


In [11]:
# text group: product name + keywords + brand + colour (brand/colour were soft signals)
df["text_blob"] = (
    df["product_name"].fillna("") + " " +
    df["meta_keywords"].fillna("") + " " +
    df["brand"] + " " +
    df["colour"]
).str.strip()

df["cat_blob"] = df["child_category"].fillna("").str.strip()

print("text_blob examples:")
for val in df["text_blob"].head(3):
    print(f"  {str(val)[:100]}")

print("\ncat_blob examples (should show subcategory labels, not brand names):")
for val in df["cat_blob"].head(5):
    print(f"  {val}")

print(f"\nEmpty text_blob: {(df['text_blob'] == '').sum():,}")
print(f"Empty cat_blob:  {(df['cat_blob'] == '').sum():,}")


text_blob examples:
  LA' Facon Cotton Kalamkari Handblock Saree Blouse Fabric 100 cms Black Base Dancers (Cotton) LA' Fac
  Sf Jeans By Pantaloons Men's Plain Slim fit T-Shirt Sf Jeans By Pantaloons Men's Plain Slim fit T-Sh
  LOVISTA Cotton Gota Patti Tassel Traditional Printed Kurti with Pant,Angrakha Salwar Suit LOVISTA Co

cat_blob examples (should show subcategory labels, not brand names):
  WomensKurtasKurtis
  MensT_Shirts
  WomensKurtasKurtis
  MensT_Shirts
  MensTracksuits

Empty text_blob: 0
Empty cat_blob:  5,149


In [12]:
if df.index.name != "uniq_id" and "uniq_id" in df.columns:
    df = df.set_index("uniq_id")

feature_cols = [
    "sales_price", "rating", "weight", "no__of_reviews", "no__of_sellers",
    "discount_percentage", "sales_rank_in_parent_category",
    "sales_rank_in_child_category", "no__of_offers",
    "amazon_prime__y_or_n", "best_seller_tag__y_or_n",
    "brand", "colour", "child_category", "text_blob", "cat_blob", "category"
]
feature_cols = [c for c in feature_cols if c in df.columns]

print(f"{'Column':<25} {'Null %':<10} {'Notes'}")
print("-" * 65)
for col in feature_cols:
    null_pct = df[col].isna().mean()
    note = "→ excluded by all-NaN guard" if null_pct == 1.0 else ""
    print(f"{col:<25} {null_pct:<10.1%} {note}")


Column                    Null %     Notes
-----------------------------------------------------------------
sales_price               9.6%       
rating                    0.0%       
weight                    100.0%     → excluded by all-NaN guard
no__of_reviews            88.5%      
no__of_sellers            96.6%      
discount_percentage       100.0%     → excluded by all-NaN guard
sales_rank_in_parent_category 15.0%      
sales_rank_in_child_category 17.2%      
no__of_offers             96.6%      
amazon_prime__y_or_n      0.0%       
best_seller_tag__y_or_n   0.0%       
brand                     0.0%       
colour                    0.0%       
child_category            0.0%       
text_blob                 0.0%       
cat_blob                  0.0%       
category                  0.0%       


In [13]:
print(f"\nClean DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Index (uniq_id): {df.index.nunique():,} unique products")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


Clean DataFrame: 30,000 rows × 36 columns
Index (uniq_id): 30,000 unique products
Memory: 114.4 MB
